## 00. 프로젝트 목적
- 본 프로젝트는 Pinecone vector store 업로드를 위한 프로젝트입니다.
- 한번에 한 파일만 올리는 것이 좋으며, namespace & metadata 전략을 적절히 사용하는 것이 유리합니다.

- [Pinecone 공식 홈페이지](https://docs.pinecone.io/integrations/langchain)
- [Pinecone 랭체인](https://python.langchain.com/v0.2/docs/integrations/vectorstores/pinecone/)

### 필요한 환경변수 로드

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv
import os

# API 키 정보 로드
load_dotenv()

# API 키 읽어오기
openai_api_key = os.environ.get('OPENAI_API_KEY')
pinecone_api_key = os.environ.get("PINECONE_API_KEY")

## 01. PDF to RAG Chunks

- `load_and_process_docs`: PDF 청킹 함수
    - 필요한 메타데이터만 추출하여 함수로 저장함.
        - headings: 추출한 객체의 대주제
        - doc_items_labels: text인지, list인지, table인지 구분
        - filename: 업로드 한 파일명

In [29]:
from langchain_docling import DoclingLoader
from docling.chunking import HybridChunker
from langchain_docling.loader import ExportType
from langchain_core.documents import Document

# 문서 로딩 및 청킹을 위한 설정
FILE_PATH = [r"C:\Users\moonk\OneDrive\바탕 화면\대학생활\세미나\고교학점제 자료\편집본\2022개정교육과정고등학교보통교과과목안내자료(경남교육).pdf"] 
EXPORT_TYPE = ExportType.DOC_CHUNKS # 문서 청크 추출 타입 설정
EMBED_MODEL_ID = "BAAI/bge-m3"

def load_and_process_docs(file_path, embed_model_id, chunk_size=1000, overlap=100):
    """
    문서를 로드하고 Pinecone 호환 메타데이터로 처리하는 함수
    
    Args:
        file_path (list): 로드할 파일 경로 리스트
        embed_model_id (str): 임베딩 모델 ID
        chunk_size (int, optional): 청크 크기. 기본값 1200
        overlap (int, optional): 청크 오버랩. 기본값 100
        
    Returns:
        list: Pinecone 호환 메타데이터를 가진 Document 객체 리스트
    """
    # DoclingLoader 초기화 및 로딩
    loader = DoclingLoader(
        file_path=file_path,
        export_type=ExportType.DOC_CHUNKS,
        chunker=HybridChunker(tokenizer=embed_model_id, chunk_size=chunk_size, overlap=overlap)
    )
    
    # 문서 로드
    original_docs = loader.load()
    print(f"로드된 문서 수: {len(original_docs)}")
    
    # 메타데이터 변환 및 새 문서 생성
    pinecone_docs = []
    
    for doc in original_docs:
        # 메타데이터에서 필요한 정보만 추출
        simplified = {}
        metadata = doc.metadata
        
        # 1. dl_meta의 doc_items의 label 추출
        if 'dl_meta' in metadata and isinstance(metadata['dl_meta'], dict):
            # headings 추출
            if 'headings' in metadata['dl_meta']:
                simplified['headings'] = metadata['dl_meta']['headings']
                
            # doc_items의 label 추출
            if 'doc_items' in metadata['dl_meta'] and metadata['dl_meta']['doc_items']:
                items = metadata['dl_meta']['doc_items']
                labels = [item.get('label') for item in items if 'label' in item]
                if labels:
                    simplified['doc_items_labels'] = labels

        # 2. origin의 filename 추출
        if ('dl_meta' in metadata and 'origin' in metadata['dl_meta'] and 
                'filename' in metadata['dl_meta']['origin']):
            simplified['filename'] = metadata['dl_meta']['origin']['filename']
        
        # 페이지 번호 추가 (있을 경우)
        if 'page' in metadata:
            simplified['page'] = metadata['page']
        
        # 새 Document 객체 생성
        pinecone_docs.append(
            Document(
                page_content=doc.page_content,
                metadata=simplified
            )
        )
    
    # 결과 요약
    if pinecone_docs:
        print(f"변환된 문서 수: {len(pinecone_docs)}")
        print("샘플 메타데이터:")
        print(pinecone_docs[0].metadata)
    
    return pinecone_docs

In [2]:
#한 페이지 씩 문서 추출
#과목 pdf 문서 한페이지씩 나누기 위해 사용용
import fitz  # PyMuPDF
from langchain.schema import Document


def extract_by_page(pdf_path):
    doc = fitz.open(pdf_path)
    docs = []
    for i, page in enumerate(doc):
        text = page.get_text()
        docs.append(Document(
            page_content=text,
            metadata={"page": i + 1, "filename": pdf_path.split('/')[-1]}
        ))
    return docs

In [3]:
#기존 함수에서 load를 위에 extract_by_page 함수 출력값으로 대체
#chunker를 by_page=True로 설정하여 페이지 단위로 청킹, chunk_size를 99999로 설정하여 페이지 전체를 하나의 청크로 처리, overlap을 0으로 설정하여 페이지 간 중복을 방지

from langchain_docling import DoclingLoader
from docling.chunking import HybridChunker
from langchain_docling.loader import ExportType
from langchain_core.documents import Document

# 문서 로딩 및 청킹을 위한 설정
EXPORT_TYPE = ExportType.DOC_CHUNKS # 문서 청크 추출 타입 설정
EMBED_MODEL_ID = "BAAI/bge-m3"

def load_and_process_docs_chunk_by_page(docs, embed_model_id, chunk_size=99999, overlap=0):
    """
    문서를 로드하고 Pinecone 호환 메타데이터로 처리하는 함수
    
    Args:
        docs (list): 한 페이지 씩 나누어진 문서 리스트
        embed_model_id (str): 임베딩 모델 ID
        chunk_size (int, optional): 청크 크기. 기본값 99999
        overlap (int, optional): 청크 오버랩. 기본값 0
        by_page (bool, optional): 페이지 단위로 청킹 여부. 기본값 True
        
    Returns:
        list: Pinecone 호환 메타데이터를 가진 Document 객체 리스트
    """
    # DoclingLoader 초기화 및 로딩
    loader = DoclingLoader(
        docs,
        export_type=ExportType.DOC_CHUNKS,
        chunker=HybridChunker(tokenizer=embed_model_id, chunk_size=chunk_size, overlap=overlap, by_page=True)   
    )
    
    # 문서 로드
    original_docs =docs #이 부분을 수정하여 extract_by_page 함수의 결과를 사용

    print(f"로드된 문서 수: {len(original_docs)}")
    
    # 메타데이터 변환 및 새 문서 생성
    pinecone_docs = []
    
    for doc in original_docs:
        # 메타데이터에서 필요한 정보만 추출
        simplified = {}
        metadata = doc.metadata
        
        # 1. dl_meta의 doc_items의 label 추출
        if 'dl_meta' in metadata and isinstance(metadata['dl_meta'], dict):
            # headings 추출
            if 'headings' in metadata['dl_meta']:
                simplified['headings'] = metadata['dl_meta']['headings']
                
            # doc_items의 label 추출
            if 'doc_items' in metadata['dl_meta'] and metadata['dl_meta']['doc_items']:
                items = metadata['dl_meta']['doc_items']
                labels = [item.get('label') for item in items if 'label' in item]
                if labels:
                    simplified['doc_items_labels'] = labels

        # 2. origin의 filename 추출
        if ('dl_meta' in metadata and 'origin' in metadata['dl_meta'] and 
                'filename' in metadata['dl_meta']['origin']):
            simplified['filename'] = metadata['dl_meta']['origin']['filename']
        
        # 페이지 번호 추가 (있을 경우)
        if 'page' in metadata:
            simplified['page'] = metadata['page']
        
        # 새 Document 객체 생성
        pinecone_docs.append(
            Document(
                page_content=doc.page_content,
                metadata=simplified
            )
        )
    
    # 결과 요약
    if pinecone_docs:
        print(f"변환된 문서 수: {len(pinecone_docs)}")
        print("샘플 메타데이터:")
        print(pinecone_docs[0].metadata)
    
    return pinecone_docs

In [4]:
# 함수 호출 및 문서 로딩
chunked_docs = load_and_process_docs(FILE_PATH,EMBED_MODEL_ID)


NameError: name 'load_and_process_docs' is not defined

In [5]:
#과목 pdf의 경우 이 코드 활용
pdf_path = r"C:\Users\moonk\OneDrive\바탕 화면\페이지별 추출 테스트용3.pdf"
docs = extract_by_page(pdf_path)

# 3. 페이지별 문서를 청크 단위로 처리 (이미 로드된 문서에서)
chunked_docs = load_and_process_docs_chunk_by_page(docs, EMBED_MODEL_ID)

로드된 문서 수: 3
변환된 문서 수: 3
샘플 메타데이터:
{'page': 1}


In [6]:
chunked_docs

[Document(metadata={'page': 1}, page_content='013\n공통국어1\n초등학교 및 중학교 ‘국어’의 성격을 계승하여 국어 관련 학문의 기본적 내용을 학습하는 과목이자 고등 \n학교 선택 과목에서의 심화 학습을 위한 토대를 마련하는 공통 과목으로 국어과의 하위 영역인 듣기·\n말하기, 읽기, 쓰기, 문법, 문학, 매체로 구성됨. 미래 사회에 대비할 수 있는 기본 역량을 함양하고, 대학 \n진학 후의 학문 활동이나 사회 진출 후의 직업 활동에 바탕이 되는 국어 능력을 기르는 데 목적이 있음.\n공통\n과목 \n영역\n핵심 아이디어\n내용 요소 \n듣기·말하기\n•\u200a듣기·말학기의 본질\n•\u200a듣기·말하기의 맥락, 목적, 담화 유형\n•\u200a의사소통 과정 및 문제해결 전략\n•\u200a듣기·말하기의 태도 \n•\u200a상황 맥락과 사회·문화적 맥락\n•\u200a대화\n•\u200a토론\n읽기\n•\u200a읽기의 본질\n•\u200a읽기 맥락, 목적\n•\u200a읽기 과정의 점검·조정, 읽기 전략\n•\u200a읽기의 태도\n•\u200a사회·문화적 맥락\n•\u200a인문, 예술, 사회, 문화, 과학, 기술 등 다양한 분야의 글\n•\u200a다양한 설명 방법을 활용하여 주제를 제시한 글\n•\u200a다양한 논증 방법을 활용하여 주장을 제시한 글\n•\u200a생각과 감정이 함축적이고 복합적으로 제시된 글\n쓰기\n•\u200a쓰기의 본질\n•\u200a쓰기 맥락, 목적\n•\u200a쓰기 전략\n•\u200a쓰기의 태도\n•\u200a사회·문화적 맥락\n•\u200a사회적 쟁점에 대한 자신의 견해를 드러내는 글\n•\u200a개성이 드러나는 글\n문법\n•\u200a문법의 본질\n•\u200a국어의 특성\n•\u200a국어 자료의 활용\n•\u200a국어 사용의 태도\n•\u200a언어 공동체의 다변화에 따른 언어\n•\u200a음운 변동\n•\u200a글과 담화에 나타난 문법 요소 및 어휘의 특성과

In [ ]:
chunked_docs[1].page_content

'내용체계\n듣기·말하기, 핵심아이디어 = • 듣기·말학기의본질 • 듣기·말하기의맥락,목적,담화유형 • 의사소통과정및문제해결전략 • 듣기·말하기의태도. 듣기·말하기, 내용요소 = • 상황맥락과사회·문화적맥락 • 대화 • 토론. 읽기, 핵심아이디어 = • 읽기의본질 • 읽기맥락,목적 • 읽기과정의점검·조정,읽기전략 • 읽기의태도. 읽기, 내용요소 = • 사회·문화적맥락 • 인문, 예술, 사회, 문화, 과학, 기술등다양한분야의글 • 다양한설명방법을활용하여주제를제시한글 • 다양한논증방법을활용하여주장을제시한글 • 생각과감정이함축적이고복합적으로제시된글. 쓰기, 핵심아이디어 = • 쓰기의본질 • 쓰기맥락,목적 • 쓰기전략 • 쓰기의태도. 쓰기, 내용요소 = • 사회·문화적맥락 • 사회적쟁점에대한자신의견해를드러내는글 • 개성이드러나는글. 문법, 핵심아이디어 = • 문법의본질 • 국어의특성 • 국어자료의활용 • 국어사용의태도. 문법, 내용요소 = • 언어공동체의다변화에따른언어 • 음운변동 • 글과담화에나타난문법요소및어휘의특성과사용. 문학, 핵심아이디어 = • 문학의본질 • 문학의소통 • 문학의수용과생산 • 문학과태도. 문학, 내용요소 = • 서정, 서사, 극, 교술 • 작가맥락,독자맥락,사회·문화적맥락,문학사적맥락. 매체, 핵심아이디어 = • 매체의본질 • 매체의수용과생산 • 매체와태도. 매체, 내용요소 = • 사회·문화적매락 • 다양한유형의매체자료\n관련정보'

In [27]:
for i, doc in enumerate(chunked_docs):
    content_bytes = len(doc.page_content.encode("utf-8"))
    if content_bytes > 4000000:
        print(f"[경고] {i}번 청크가 너무 큽니다: {content_bytes / 1024:.2f} KB")

- `extract_and_export_tables`: 테이블 추출 함수
    - PDF에 있는 테이블을 csv로 저장하는 함수

In [30]:
import time
from pathlib import Path
import pandas as pd

from docling.document_converter import DocumentConverter

def extract_and_export_tables(input_path, output_directory):
    """
    PDF 파일에서 테이블을 추출하고 CSV로 저장하는 함수
    
    Args:
        input_path (str): 입력 PDF 파일 경로
        output_directory (str): 출력 CSV 파일을 저장할 디렉토리
    
    Returns:
        int: 추출한 표의 수
    """
    # 출력 디렉토리를 Path 객체로 변환
    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)  # 디렉토리 생성
    
    # 입력 파일 경로를 Path 객체로 변환
    input_path = Path(input_path)
    doc_filename = input_path.stem  # 파일명에서 확장자 제외한 부분
    
    # 문서 변환기 초기화
    doc_converter = DocumentConverter()
    
    # 변환 시작 시간 기록
    start_time = time.time()
    print(f"문서 변환 시작: {input_path}")
    
    # 문서 변환
    conv_res = doc_converter.convert(str(input_path))
    
    # 표 내보내기
    for table_ix, table in enumerate(conv_res.document.tables):
        table_df = table.export_to_dataframe()
        print(f"## Table {table_ix}")
        print(table_df.to_markdown(index=False))  # 표를 마크다운 형식으로 출력
        
        # 표를 csv로 저장
        csv_filename = output_directory / f"{doc_filename}-table-{table_ix+1}.csv"
        print(f"CSV 파일 저장: {csv_filename}")
        table_df.to_csv(csv_filename, index=False,encoding='utf-8-sig')
            
    # 변환 소요 시간 출력
    elapsed_time = time.time() - start_time
    print(f"변환 소요 시간: {elapsed_time:.2f}초")
    
    # 추출한 표 수 확인
    table_count = len(conv_res.document.tables)
    print(f"추출한 표 수: {table_count}")
        
    return table_count
    
# 표 추출 및 내보내기 함수 호출
OUTPUT_DIR = '../create_dataset/source_data/tables'
INPUT_PATH = r"C:\Users\moonk\OneDrive\바탕 화면\페이지별 추출 테스트용3.pdf"
try:
    table_count = extract_and_export_tables(INPUT_PATH, OUTPUT_DIR)
    print(f"총 {table_count}개의 표가 추출되었습니다.")
except Exception as e:
    print(f"오류 발생: {e}")

문서 변환 시작: C:\Users\moonk\OneDrive\바탕 화면\페이지별 추출 테스트용3.pdf
## Table 0
| 영역        | 핵심아이디어                                                                                           | 내용요소                                                                                                                                                                                            |
|:------------|:-------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| 듣기·말하기 | • 듣기·말학기의본질 • 듣기·말하기의맥락,목적,담화유형 • 의사소통과정및문제해결전략 • 듣기·말하기의태도 | • 상황맥락과사회·문화적맥락 • 대화 • 토론                                                                                                                                                           |
| 읽기        | • 읽기의본질 • 읽기맥락,목적 • 읽기과정의점검·조정,읽기전략 

In [9]:
#현재 디렉토리 확인
import os
print("현재 디렉토리:", os.getcwd())

현재 디렉토리: c:\Users\moonk\OneDrive\바탕 화면\대학생활\세미나\프로젝트\bearable_chatbot\vector_store


## 02. Pinecone Upload
- Index 생성
- web에서 직접 index 생성하는 것을 추천함.
    - https://app.pinecone.io/
- 문서를 올릴 때는 반드시 한 문서씩 업로드 (청킹 확인 후 업로드 권장)

In [36]:
from pinecone import Pinecone

# Pinecone API 키와 환경 설정
pc = Pinecone(api_key=pinecone_api_key)
pc_index = pc.Index("myfolio-chatbot")

In [37]:
# Pinecone 인덱스 확인
pc.list_indexes()

[
    {
        "name": "myfolio-chatbot",
        "metric": "cosine",
        "host": "myfolio-chatbot-v2dkd4p.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 3072,
        "deletion_protection": "disabled",
        "tags": {
            "embedding_model": "text-embedding-3-large"
        }
    }
]

In [42]:
from langchain.embeddings import OpenAIEmbeddings

# OpenAI 임베딩 인스턴스 생성
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-large',
    openai_api_key=openai_api_key
)

C:\Users\moonk\AppData\Local\Temp\ipykernel_16396\588479685.py:4: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(


In [ ]:
docs = chunked_docs

이부분 뭐더라??

In [ ]:
from tqdm import tqdm  # 진행 상황을 보여주기 위한 라이브러리
from langchain_pinecone import PineconeVectorStore


# PineconeVectorStore 인스턴스 생성 및 문서 추가
pinecone_database = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name="myfolio-chatbot",
    namespace="curriculum", # 네임스페이스 설정: 고교학점제 -> curriculum, 진로&진학 상담 -> course, 서비스 문의 -> service
)

In [53]:
from langchain_pinecone import PineconeVectorStore
from tqdm import tqdm

docs = chunked_docs
def upload_in_batches(docs, embeddings, index_name, namespace, batch_size=32):
    all_batches = [docs[i:i+batch_size] for i in range(0, len(docs), batch_size)]
    for i, batch in enumerate(tqdm(all_batches, desc="Uploading to Pinecone")):
        PineconeVectorStore.from_documents(
            documents=batch,
            embedding=embeddings,
            index_name=index_name,
            namespace=namespace,
        )

# 실행
upload_in_batches(
    docs=docs,
    embeddings=embeddings,
    index_name="myfolio-chatbot",
    namespace="curriculum",
    batch_size=32  # 필요시 16~64 등으로 조절 가능
)


Uploading to Pinecone: 100%|██████████| 1/1 [00:04<00:00,  4.87s/it]


In [54]:
# 생성된 객체에 대해 문서 추가
pinecone_database.add_documents(
    documents=docs,
    index_name="myfolio-chatbot",
    namespace="curriculum-table", # 네임스페이스 설정: 고교학점제 -> curriculum, 진로&진학 상담 -> course, 서비스 문의 -> service
)

['207575e1-7a90-4f37-8439-e21089ee544d',
 '250123ab-c5f6-41a5-ad5d-3f3f4472a983',
 '9ced1f9c-08c5-4e2a-8422-63becbc4ef2e']

## 03. Pinecone Delete

아래 코드는 반드시 필요한 경우에만 사용합니다.
- 파인콘에 저장한 문서들을 모두 삭제하는 코드

In [ ]:
pc = Pinecone(api_key=pinecone_api_key)
index = pc.Index("myfolio-chatbot")

# 특정 네임스페이스의 모든 레코드 삭제
#index.delete(delete_all=True, namespace="curriculum-table")

{}